# ComfyUI de Leo — Z-Image Turbo (Colab)

Notebook propio para correr ComfyUI con Z-Image Turbo + ControlNet + upscale, manejado desde **comfyweb**.

**Cómo usarlo:** arriba, *Entorno de ejecución → Ejecutar todo*. La 1ª vez baja los modelos a tu Google Drive (~14 GB, tarda); después ya quedan y arranca en ~2-3 min.

Al final imprime una **URL pública** — pegála en comfyweb (botón *Colab (Cloudflare)*).

> Para una URL **fija** (que no cambie cada vez) ver `README.md` → sección *Túnel fijo*.

## 1) Montar Google Drive (los modelos viven acá, no se rebajan)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

# Carpeta tuya en Drive (se crea sola la 1ª vez)
DRIVE_BASE = '/content/drive/MyDrive/ComfyUI-Leo'
MODELS = os.path.join(DRIVE_BASE, 'models')
for sub in ['diffusion_models','text_encoders','vae','model_patches','upscale_models','loras']:
    os.makedirs(os.path.join(MODELS, sub), exist_ok=True)
print('Drive montado. Modelos en:', MODELS)

## 2) Clonar ComfyUI + custom nodes (Z-Image + ControlNet + upscale)

In [ ]:
import os
%cd /content
if not os.path.isdir('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt

# Custom nodes que usa el pipeline DIPLO (ControlNet preprocesadores, tiles, etc.)
CN = '/content/ComfyUI/custom_nodes'
repos = [
    'https://github.com/ltdrdata/ComfyUI-Manager',
    'https://github.com/Fannovel16/comfyui_controlnet_aux',   # AIO_Preprocessor
    'https://github.com/TTPlanetPig/Comfyui_TTP_Toolset',     # tiles
    'https://github.com/yolain/ComfyUI-Easy-Use',
    'https://github.com/rgthree/rgthree-comfy',
    'https://github.com/LAOGOU-666/Comfyui-Memory_Cleanup',   # RAMCleanup
]
for u in repos:
    name = u.rstrip('/').split('/')[-1]
    p = os.path.join(CN, name)
    if not os.path.isdir(p):
        !git clone --depth 1 {u} {p}
    req = os.path.join(p, 'requirements.txt')
    if os.path.isfile(req):
        !pip install -q -r {req}
print('\nComfyUI + custom nodes listos')

## 3) Descargar modelos a Drive (solo la 1ª vez)

Si ya están en tu Drive, los saltea. Fuentes oficiales (Apache-2.0).

In [ ]:
import os

def dl(url, dest, min_mb=1):
    if os.path.isfile(dest) and os.path.getsize(dest) > min_mb*1_000_000:
        print('✓ ya está:', os.path.basename(dest)); return
    print('↓ bajando:', os.path.basename(dest))
    !wget -q --show-progress -O "{dest}" "{url}"

MODELS = '/content/drive/MyDrive/ComfyUI-Leo/models'

# Difusión (Z-Image Turbo fp8 e4m3fn, ideal para T4)
dl('https://huggingface.co/T5B/Z-Image-Turbo-FP8/resolve/main/z-image-turbo-fp8-e4m3fn.safetensors',
   f'{MODELS}/diffusion_models/z-image-turbo-fp8-e4m3fn.safetensors', 500)
# Text encoder (Qwen3-4B fp8) — CLIPLoader tipo lumina2
dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b_fp8_mixed.safetensors',
   f'{MODELS}/text_encoders/qwen_3_4b_fp8_mixed.safetensors', 500)
# VAE
dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors',
   f'{MODELS}/vae/ae.safetensors', 50)
# ControlNet Union (Canny/Depth/Pose/etc.)
dl('https://huggingface.co/alibaba-pai/Z-Image-Turbo-Fun-Controlnet-Union/resolve/main/Z-Image-Turbo-Fun-Controlnet-Union.safetensors',
   f'{MODELS}/model_patches/Z-Image-Turbo-Fun-Controlnet-Union.safetensors', 500)
# Upscaler 2x
dl('https://huggingface.co/Kim2091/2x-AnimeSharpV4/resolve/main/2x-AnimeSharpV4_RCAN.safetensors',
   f'{MODELS}/upscale_models/2x-AnimeSharpV4_RCAN.safetensors', 5)

print('\nModelos OK en Drive')

## 4) Apuntar ComfyUI a los modelos de Drive + lanzar + túnel público

Al final imprime la **URL pública**. Pegála en comfyweb.

In [ ]:
# ComfyUI lee los modelos desde Drive (no copia, los usa en su lugar)
yaml = '''leo_drive:
  base_path: /content/drive/MyDrive/ComfyUI-Leo/models
  diffusion_models: diffusion_models
  text_encoders: text_encoders
  clip: text_encoders
  vae: vae
  model_patches: model_patches
  upscale_models: upscale_models
  loras: loras
'''
open('/content/ComfyUI/extra_model_paths.yaml','w').write(yaml)
print('extra_model_paths.yaml escrito')

!pip install -q pycloudflared
import threading, time, socket
%cd /content/ComfyUI

def tunnel():
    # espera a que ComfyUI levante el 8188
    while socket.socket().connect_ex(('127.0.0.1', 8188)) != 0:
        time.sleep(1)
    from pycloudflared import try_cloudflare
    url = try_cloudflare(port=8188).tunnel
    print('\n\n' + '='*48)
    print('  URL PUBLICA (pegala en comfyweb):')
    print('  ' + url)
    print('='*48 + '\n')

threading.Thread(target=tunnel, daemon=True).start()
!python main.py --dont-print-server

---
### (Opcional) Túnel FIJO con tu dominio

Para que la URL **no cambie** cada vez, en vez de la celda 4 usá un Cloudflare **named tunnel** con tu dominio. Ver pasos en `README.md`. Una vez configurado, reemplázás la celda 4 por:

```python
%cd /content/ComfyUI
import threading
open('/content/ComfyUI/extra_model_paths.yaml','w').write(yaml)
# pega tu token de tunnel de Cloudflare (Colab > Secretos > CF_TUNNEL_TOKEN)
from google.colab import userdata
TOKEN = userdata.get('CF_TUNNEL_TOKEN')
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
threading.Thread(target=lambda: __import__('subprocess').run(['cloudflared','tunnel','run','--token',TOKEN]), daemon=True).start()
!python main.py --dont-print-server
```

Tu URL fija será algo como `https://comfy.leoblumfeld.com` (la definís vos en Cloudflare).